In [9]:
# Drive 마운트 해제 후 재마운트
drive.flush_and_unmount()

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [10]:
import subprocess
result = subprocess.run(['find', '/content/drive/MyDrive', '-iname', '*resnet50*'],
                        capture_output=True, text=True, timeout=120)
print(result.stdout if result.stdout else "여전히 없음")

여전히 없음


In [11]:
import os
print(os.listdir('/content/drive/MyDrive'))

['10824 현수빈.mp4', 'Classroom', 'custom_data.py의 사본', '유스케이스 명세서_최종.gdoc', '소웨공 과제.gslides', '제목 없는 프레젠테이션.gslides', '유스케이스 명세서.gdoc', '딥러닝', 'PR202310888', 'IMG_5164.jpeg', 'IMG_5165.jpeg', 'project23', 'dfdc_train_part_2', '인간중심 UI UX 디자인 사례 연구_Waymo.gdoc', 'processed_frames_v2', 'data', 'best_deepfake_model.pth', 'dfdc_part_0_test_results.csv', 'metadata.json', 'TEST_3', 'Model2.ipynb', 'Web_test1.ipynb']


In [12]:
import os
print(os.listdir('/content/drive/MyDrive/TEST_3'))

['checkpoints', 'data', 'processed_frames_v2', 'content', 'gradcam_samples.png', 'gradcam_fn_analysis.png', 'gradcam_tp_analysis.png', 'padding_comparison.png', 'processed_frames_v3', 'v3_threshold_analysis.png', 'v3_gradcam_tp.png', 'v3_gradcam_fn.png', 'v3_gradcam_tn.png', 'v3_gradcam_fp.png', 'v2_masking_example.png', 'TEST_3.ipynb', '녹음 2026-05-27 194029.mp4', '전문가2.mp4', 'custom_video_test.png', 'Model2.ipynb']


In [13]:
import os
checkpoint_dir = '/content/drive/MyDrive/TEST_3/checkpoints'
if os.path.exists(checkpoint_dir):
    print(os.listdir(checkpoint_dir))
else:
    print("checkpoints 폴더 없음")

['resnet50_frame_best.pth', 'vit_frame_best.pth', 'resnet_lstm_best.pth', 'vit_lstm_best.pth', 'resnet50_frame_v3_best.pth']


In [ ]:
!pip install mediapipe gradio -q

In [14]:
import os
import cv2
import torch
import torch.nn as nn
import numpy as np
from PIL import Image
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from torchvision import models, transforms
import gradio as gr

# ===== 설정 =====
CHECKPOINT_PATH = "/content/drive/MyDrive/TEST_3/checkpoints/resnet50_frame_v3_best.pth"  # ← 이 줄만 수정됨
MODEL_PATH = "/content/face_landmarker.task"
THRESHOLD = 0.30
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ===== Drive 마운트 =====
from google.colab import drive
drive.mount('/content/drive')

# ===== MediaPipe 얼굴 검출 모델 준비 =====
if not os.path.exists(MODEL_PATH):
    os.system(f"wget -q https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task -O {MODEL_PATH}")

base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
options = vision.FaceLandmarkerOptions(
    base_options=base_options, num_faces=1, min_face_detection_confidence=0.3
)
detector = vision.FaceLandmarker.create_from_options(options)

MOUTH_LANDMARKS = [
    61, 146, 91, 181, 84, 17, 314, 405, 321, 375, 291,
    78, 95, 88, 178, 87, 14, 317, 402, 318, 324, 308,
    13, 312, 311, 310, 415, 0, 267, 269, 270, 409,
]

# ===== 입 주변 크롭 =====
def apply_mouth_crop(frame, img_size=224, padding_ratio=0.3, detect_max_dim=480):
    h_orig, w_orig = frame.shape[:2]
    if max(h_orig, w_orig) > detect_max_dim:
        scale = detect_max_dim / max(h_orig, w_orig)
        small = cv2.resize(frame, (int(w_orig * scale), int(h_orig * scale)))
    else:
        small = frame

    rgb_small = cv2.cvtColor(small, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_small)
    results = detector.detect(mp_image)
    if not results.face_landmarks:
        return None

    landmarks = results.face_landmarks[0]
    mouth_points = np.array([
        (int(landmarks[idx].x * w_orig), int(landmarks[idx].y * h_orig))
        for idx in MOUTH_LANDMARKS
    ])
    x_min, y_min = mouth_points.min(axis=0)
    x_max, y_max = mouth_points.max(axis=0)
    box_w, box_h = x_max - x_min, y_max - y_min
    pad_x, pad_y = int(box_w * padding_ratio), int(box_h * padding_ratio)
    x1, y1 = max(0, x_min - pad_x), max(0, y_min - pad_y)
    x2, y2 = min(w_orig, x_max + pad_x), min(h_orig, y_max + pad_y)

    crop_w, crop_h = x2 - x1, y2 - y1
    if crop_w > crop_h:
        diff = crop_w - crop_h
        y1, y2 = max(0, y1 - diff // 2), min(h_orig, y2 + (diff - diff // 2))
    elif crop_h > crop_w:
        diff = crop_h - crop_w
        x1, x2 = max(0, x1 - diff // 2), min(w_orig, x2 + (diff - diff // 2))

    cropped = frame[y1:y2, x1:x2]
    if cropped.size == 0:
        return None
    return cv2.resize(cropped, (img_size, img_size))

def extract_frames(cap, fps, start_sec, duration_sec=3, num_frames=8):
    start_frame = int(fps * start_sec)
    max_frames = int(fps * duration_sec)
    interval = max(1, max_frames // num_frames)
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    frames = []
    idx = 0
    while idx < max_frames and len(frames) < num_frames:
        ret, frame = cap.read()
        if not ret:
            break
        if idx % interval == 0:
            frames.append(frame)
        idx += 1
    return frames

def try_extract(cap, fps, start_sec, duration_sec=3, num_frames=8):
    frames = extract_frames(cap, fps, start_sec, duration_sec, num_frames)
    valid = []
    for frame in frames:
        processed = apply_mouth_crop(frame)
        if processed is not None:
            valid.append(processed)
    return valid

# ===== 모델 로드 =====
print("모델 로딩 중...")
model = models.resnet50(weights=None)
model.fc = nn.Linear(model.fc.in_features, 2)
ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model = model.to(DEVICE).eval()
print(f"모델 로드 완료 (val_acc: {ckpt.get('val_acc', '정보 없음')})")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# ===== Grad-CAM =====
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None
        target_layer.register_forward_hook(lambda m, i, o: setattr(self, 'activations', o.detach()))
        target_layer.register_full_backward_hook(lambda m, gi, go: setattr(self, 'gradients', go[0].detach()))
    def __call__(self, x, class_idx=1):
        logits = self.model(x)
        self.model.zero_grad()
        logits[0, class_idx].backward()
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = torch.relu(cam).squeeze().cpu().numpy()
        if cam.max() > 0:
            cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam

gradcam = GradCAM(model, model.layer4[-1])

# ===== 영상 하나 받아서 판정하는 함수 =====
def predict_video(video_path):
    if video_path is None:
        return "영상을 업로드해주세요", None, None

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    if fps == 0 or total_frames == 0:
        return "영상을 읽을 수 없습니다", None, None
    total_sec = total_frames / fps

    mid_start = max(0, (total_sec / 2) - 1.5)
    end_start = max(0, total_sec - 3)

    best = []
    for start_sec in [mid_start, 0, end_start]:
        valid = try_extract(cap, fps, start_sec=start_sec)
        if len(valid) > len(best):
            best = valid
        if len(best) >= 8:
            break
    cap.release()

    if len(best) <= 4:
        return "⚠️ 얼굴(입)을 충분히 인식하지 못했습니다. 다른 영상을 시도해주세요.", None, None

    with torch.no_grad():
        images = [transform(Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))) for f in best]
        batch = torch.stack(images).to(DEVICE)
        probs = torch.softmax(model(batch), dim=1)[:, 1].cpu().numpy()

    mean_prob = float(probs.mean())
    verdict = "FAKE (합성 의심)" if mean_prob >= THRESHOLD else "REAL (진짜로 판단)"
    emoji = "🚨" if mean_prob >= THRESHOLD else "✅"

    most_idx = int(probs.argmax())
    target_pil = Image.fromarray(cv2.cvtColor(best[most_idx], cv2.COLOR_BGR2RGB))
    cam = gradcam(transform(target_pil).unsqueeze(0).to(DEVICE), class_idx=1)

    img_np = np.array(target_pil.resize((224, 224)))
    cam_resized = cv2.resize(cam, (224, 224))
    heatmap = cv2.applyColorMap((cam_resized * 255).astype(np.uint8), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    overlay = (0.5 * heatmap + 0.5 * img_np).astype(np.uint8)

    result_text = f"""
{emoji} 판정 결과: {verdict}

평균 FAKE 확률: {mean_prob:.1%}
임계값: {THRESHOLD:.0%}
분석한 프레임 수: {len(best)}장
"""
    return result_text, Image.fromarray(img_np), Image.fromarray(overlay)

# ===== 웹 화면 구성 및 실행 =====
demo = gr.Interface(
    fn=predict_video,
    inputs=gr.Video(label="영상을 업로드하세요"),
    outputs=[
        gr.Textbox(label="판정 결과"),
        gr.Image(label="분석한 프레임 (입 주변)"),
        gr.Image(label="Grad-CAM (모델이 주목한 부분)"),
    ],
    title="🔍 전문가 사칭 딥페이크 탐지 데모",
    description="영상을 업로드하면 입 주변 영역을 분석하여 딥페이크 여부를 판정합니다.",
)

demo.launch(share=True, debug=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
모델 로딩 중...
모델 로드 완료 (val_acc: 0.9152677857713829)
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://df94b67d75e88f191c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://df94b67d75e88f191c.gradio.live
